In [1]:
import numpy as np
import json
import torch
from torch import optim
import torchvision.transforms as transforms
import torch.nn as nn
from PIL import Image
import cv2
from torchvision.io import decode_image
from torch.utils.data import Dataset, DataLoader
import random
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, confusion_matrix
from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights
import torch.nn.functional as F
import string
from torch.nn.utils.rnn import pad_sequence

In [2]:
device = torch.device('cuda')
BATCH_SIZE = 16
LEARNING_RATE = 0.001
IMG_WDTH=450
IMG_HGHT=300

In [5]:
with open('PhotoBook/v2/gold-extracted.json', 'r') as file:
    data = json.load(file)
    img_dsrptn_tpls = []
    char2indx = {"<pad>": 0}
    counter = 1
    translator = str.maketrans('', '', string.punctuation)
    for key in data.keys():
        img_cv2array = cv2.imread("PhotoBook/images/" + key)
        
        img_cv2array = cv2.cvtColor(img_cv2array, cv2.COLOR_BGR2RGB)

        img_cv2array = cv2.resize(img_cv2array, (IMG_WDTH, IMG_HGHT), interpolation=cv2.INTER_AREA)
        img_pytorch_tensor = torch.from_numpy(img_cv2array)
        img_pytorch_tensor_prmtd = img_pytorch_tensor.permute(2, 0, 1)
        
        for subkey in data[key]:
            for dictionary_object in data[key][subkey]:
                for char in dictionary_object["Message_Text"]:
                    lower = char.lower()
                    if lower not in char2indx:
                        char2indx[lower] = counter
                        counter += 1
                        
        for subkey in data[key]:
            for dictionary_object in data[key][subkey]:
                dscrptn_tensor = torch.tensor([char2indx[ch.lower()] for ch in dictionary_object["Message_Text"]], dtype=torch.long)
                img_dsrptn_tpls.append((img_pytorch_tensor_prmtd, dscrptn_tensor))

In [6]:
vocabulary_length = len(char2indx)

In [8]:
def split_list(big_list, train_ratio=0.8, test_ratio=0.1, dev_ratio=0.1):
    
    random.seed(42)
    random.shuffle(big_list)

    total_len = len(big_list)
    intg_train = int(total_len*train_ratio)
    intg_dev = int(total_len*dev_ratio)

    
    train_list = big_list[:intg_train]
    dev_list = big_list[intg_train:(intg_dev+intg_train)]
    test_list = big_list[(intg_train+intg_dev):]

    return train_list, dev_list, test_list

train_list, dev_list, test_list = split_list(img_dsrptn_tpls)

In [9]:
class raw_DS(Dataset):
    def __init__(self, main_list, transform=None):
        self.main_list = main_list
        self.transform = transform

    def __len__(self):
        return len(self.main_list)
        
    def __getitem__(self, idx):
        img, description = self.main_list[idx][0], self.main_list[idx][1]
        
        return img, description

In [10]:
train_raw_DS = raw_DS(train_list)
dev_raw_DS = raw_DS(dev_list)
test_raw_DS = raw_DS(test_list)

In [11]:
def collate_fn(batch):
    images, texts = zip(*batch)
    images = torch.stack(images)
    texts = [torch.tensor(t, dtype=torch.long) for t in texts]
    texts = pad_sequence(texts, batch_first=True, padding_value=0)
    return images, texts

In [32]:
train_dataloader = DataLoader(dataset=train_raw_DS,
                              batch_size=BATCH_SIZE,
                              shuffle=True,
                              collate_fn=collate_fn
                             )

dev_dataloader = DataLoader(dataset=dev_raw_DS,
                            batch_size=BATCH_SIZE,
                            collate_fn=collate_fn
                           )

test_dataloader = DataLoader(dataset=test_raw_DS,
                            batch_size=BATCH_SIZE,
                             collate_fn=collate_fn
                            )


In [14]:
class ImageEncoder(nn.Module):
    def __init__(self, embed_dim=256):
        super().__init__()
        weights = ResNet18_Weights.DEFAULT
        self.backbone = resnet18(weights=weights)
        self.backbone.fc = nn.Identity()
        self.projection = nn.Linear(512, embed_dim)
        self.transforms = weights.transforms()
        
    def forward (self, x):
        x = self.transforms(x)
        features = self.backbone(x)
        emb = self.projection(features)
        return F.normalize(emb, dim=1)

class TextEncoder(nn.Module):
    def __init__(self, vocab_size, embed_dim=256):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, 512)
        encoder_layer = nn.TransformerEncoderLayer(d_model=512, nhead=8, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.projection = nn.Linear (512, embed_dim)
        
    def forward(self, x):
        x = self.embedding(x)
        x = self.transformer(x)
        x = x.mean(dim=1)
        x = self.projection(x)
        return(F.normalize(x, dim=1))
        
class Model_bl(nn.Module):
    def __init__(self, vocab_size, embed_dim=256):
        super().__init__()
        self.image_encoder = ImageEncoder(embed_dim)
        self.text_encoder = TextEncoder(vocab_size, embed_dim)
        self.temperature = nn.Parameter(torch.tensor(0.07))
        
    def forward(self, images, texts):
        img_emb = self.image_encoder(images)
        txt_emb = self.text_encoder(texts)
        logits = txt_emb @ img_emb.T / self.temperature
        return logits

In [15]:
baseline_model = Model_bl(vocabulary_length).to(device)

In [17]:
optimizer = optim.AdamW(baseline_model.parameters(), lr=LEARNING_RATE)

In [16]:
def contrastive_loss(logits):
    B = logits.size(0)
    labels = torch.arange(B, device=logits.device)
    loss_t2i = F.cross_entropy(logits, labels)
    loss_i2t = F.cross_entropy(logits.T, labels)
    return (loss_t2i + loss_i2t) / 2

In [20]:
def evaluate(model, val_loader, device):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for images, texts in val_loader:
            images = images.to(device)
            texts = texts.to(device)

            logits = model(images, texts)
            loss = contrastive_loss(logits)

            total_loss += loss.item()

    return total_loss / len(val_loader)

def train(model, train_loader, val_loader, optimizer, device, epochs=10):
    model.to(device)

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1} [train]")

        for images, texts in pbar:
            images = images.to(device)
            texts = texts.to(device)

            logits = model(images, texts)
            loss = contrastive_loss(logits)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            pbar.set_postfix(loss=loss.item())
            
        train_loss = total_loss / len(train_loader)
        val_loss = evaluate(model, val_loader, device)
        print(f"Train loss: {train_loss:.4f}")
        print(f"Val loss: {val_loss:.4f}")

In [95]:
train(baseline_model, train_dataloader, dev_dataloader, optimizer, device, epochs=35)

Epoch 1 [train]:   0%|                             | 0/51 [00:00<?, ?it/s]/tmp/ipykernel_779025/100564079.py:5: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  texts = [torch.tensor(t, dtype=torch.long) for t in texts]
Epoch 1 [train]: 100%|█████████| 51/51 [00:02<00:00, 20.12it/s, loss=2.71]


Train loss: 2.7714
Val loss: 2.6064


Epoch 2 [train]: 100%|█████████| 51/51 [00:02<00:00, 20.25it/s, loss=2.71]


Train loss: 2.7713
Val loss: 2.6064


Epoch 3 [train]: 100%|█████████| 51/51 [00:02<00:00, 20.28it/s, loss=2.71]


...

Train loss: 2.7713
Val loss: 2.6064


Epoch 49 [train]: 100%|████████| 51/51 [00:02<00:00, 18.77it/s, loss=2.71]


Train loss: 2.7713
Val loss: 2.6064


Epoch 50 [train]: 100%|████████| 51/51 [00:02<00:00, 18.71it/s, loss=2.71]


Train loss: 2.7713
Val loss: 2.6064


In [98]:
def test_model(model, test_loader, device, top=1):
    model.eval()
    model.to(device)
    total = 0
    correct = 0
    with torch.no_grad():
        for images, texts in test_loader:
            images = images.to(device)
            texts = texts.to(device)
            logits = model(images, texts)
            if top == 1:
                preds = logits.argmax(dim=1)
                labels = torch.arange(logits.size(0), device=device)
                correct += (preds == labels).sum().item()
                total += logits.size(0)
            if top == 5:
                preds = logits.argmax(dim=1)
                top5 = logits.topk(5, dim=1).indices
                labels = torch.arange(logits.size(0), device=device).unsqueeze(1)
                correct += (top5 == labels).any(dim=1).sum().item()
                total += logits.size(0)

    acc = correct / total
    print(f"Accuracy: {acc:.4f}")

In [96]:
test_model(baseline_model, test_dataloader, device, top=5)

Top-1 Accuracy: 0.3398


/tmp/ipykernel_779025/100564079.py:5: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  texts = [torch.tensor(t, dtype=torch.long) for t in texts]


In [44]:
%store -r file2detected_objects

In [80]:
with open('PhotoBook/v2/gold-extracted.json', 'r') as file:
    data = json.load(file)
    dtctn_dsrptn_tpls2 = []
    char2indx2 = {"<pad>": 0}
    counter = 1
    translator = str.maketrans('', '', string.punctuation)
    for key in data.keys():
        concatonated = " ".join(file2detected_objects[key])
        for char in concatonated:
            lower = char.lower()
            if lower not in char2indx:
                char2indx[lower] = counter
                counter += 1
        encoded_list_of_objects = [char2indx[char.lower()] for char in concatonated]        
        objects_tpl =  torch.tensor(encoded_list_of_objects)

        for subkey in data[key]:
            for dictionary_object in data[key][subkey]:
                for char in dictionary_object["Message_Text"]:
                    lower = char.lower()
                    if lower not in char2indx:
                        char2indx[lower] = counter
                        counter += 1
                        
        for subkey in data[key]:
            for dictionary_object in data[key][subkey]:
                dscrptn_tensor = torch.tensor([char2indx[ch.lower()] for ch in dictionary_object["Message_Text"]], dtype=torch.long)
                dtctn_dsrptn_tpls2.append((objects_tpl, dscrptn_tensor))

In [81]:
vocabulary_length2 = len(char2indx)

In [83]:
train_list2, dev_list2, test_list2 = split_list(dtctn_dsrptn_tpls2)

In [85]:
class dtctrn_DS(Dataset):
    def __init__(self, main_list, transform=None):
        self.main_list = main_list
        self.transform = transform
    def __len__(self):
        return len(self.main_list)
        
    def __getitem__(self, idx):
        objects, description = self.main_list[idx][0], self.main_list[idx][1]
        
        return objects, description

In [86]:
train_dtctrn_DS = dtctrn_DS(train_list2)
dev_dtctrn_DS = dtctrn_DS(dev_list2)
test_dtctrn_DS = dtctrn_DS(test_list2)

In [88]:
def collate_fn2(batch):
    objects, texts = zip(*batch)
    objects = [torch.tensor(t, dtype=torch.long) for t in objects]
    texts = [torch.tensor(t, dtype=torch.long) for t in texts]
    objects = pad_sequence(objects, batch_first=True, padding_value=0)
    texts = pad_sequence(texts, batch_first=True, padding_value=0)

    return objects, texts

In [89]:
train_dataloader2 = DataLoader(dataset=train_dtctrn_DS,
                               batch_size=BATCH_SIZE,
                               shuffle=True,
                               collate_fn=collate_fn2
                              )

dev_dataloader2 = DataLoader(dataset=dev_dtctrn_DS,
                             batch_size=BATCH_SIZE,
                             collate_fn=collate_fn2
                             )

test_dataloader2 = DataLoader(dataset=test_dtctrn_DS,
                             batch_size=BATCH_SIZE,
                             collate_fn=collate_fn2
                             )

In [90]:
class TextEncoder2(nn.Module):
    def __init__(self, vocab_size, embed_dim=256):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, 512)
        encoder_layer = nn.TransformerEncoderLayer(d_model=512, nhead=8, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.projection = nn.Linear (512, embed_dim)
        
    def forward(self, x):
        x = self.embedding(x)
        x = self.transformer(x)
        x = x.mean(dim=1)
        x = self.projection(x)
        return(F.normalize(x, dim=1))
        
class Model_dtctrn(nn.Module):
    def __init__(self, vocab_size, embed_dim=256):
        super().__init__()
        self.text_encoder = TextEncoder2(vocab_size, embed_dim)
        self.temperature = nn.Parameter(torch.tensor(0.0))
        
    def forward(self, objects, texts):
        objcts_emb = self.text_encoder(objects)
        txt_emb = self.text_encoder(texts)
        logits = (txt_emb @ objcts_emb.T) * torch.exp(self.temperature)
        return logits

In [91]:
detectron_model = Model_dtctrn(vocabulary_length2).to(device)

In [99]:
train(detectron_model, train_dataloader2, dev_dataloader2, optimizer, device, epochs=35)

Epoch 1 [train]:   0%|                             | 0/51 [00:00<?, ?it/s]/tmp/ipykernel_779025/1925981205.py:3: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  objects = [torch.tensor(t, dtype=torch.long) for t in objects]
/tmp/ipykernel_779025/1925981205.py:4: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  texts = [torch.tensor(t, dtype=torch.long) for t in texts]
Epoch 1 [train]: 100%|█████████| 51/51 [00:01<00:00, 26.16it/s, loss=2.75]


Train loss: 2.7881
Val loss: 2.6153


Epoch 2 [train]: 100%|█████████| 51/51 [00:01<00:00, 26.29it/s, loss=2.75]


Train loss: 2.7882
Val loss: 2.6153


Epoch 3 [train]: 100%|█████████| 51/51 [00:01<00:00, 26.35it/s, loss=2.69]


...

Train loss: 2.7877
Val loss: 2.6153


Epoch 34 [train]: 100%|████████| 51/51 [00:01<00:00, 26.62it/s, loss=2.76]


Train loss: 2.7857
Val loss: 2.6153


Epoch 35 [train]: 100%|████████| 51/51 [00:01<00:00, 25.90it/s, loss=2.74]


Train loss: 2.7832
Val loss: 2.6153


In [101]:
test_model(detectron_model, test_dataloader2, device, top=5)

Accuracy: 0.0971


/tmp/ipykernel_779025/1925981205.py:3: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  objects = [torch.tensor(t, dtype=torch.long) for t in objects]
/tmp/ipykernel_779025/1925981205.py:4: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  texts = [torch.tensor(t, dtype=torch.long) for t in texts]
